加载验证过的测试样例

In [ ]:
from utils import read_jsonl,read_jsonl_gz
ds_all=read_jsonl_gz('../test-cases-verifier-from-scratch/verified-result/checked_test_cases.jsonl.gz')
# ds_all=read_jsonl('../test-cases-verifier-from-scratch/verified-result/checked_test_cases.jsonl')

构建lc_spec_ds

In [ ]:
ds_all_new=[]
for d in ds_all:
    if d['result']=='pass':
        continue
    if d['result']['status_code']!=10 and d['result']['status_code']!=15:
        continue
    d_new=dict(
        task_id=d['slug'],
        test_input_params=d['test_cases'],
        run_success=d['result']['run_success'],
    )
    if d['result']['run_success'] is True:
        if len(d['result']['code_answer'])>0:
            d_new['code_answer']=d['result']['expected_code_answer'][0]
        else:
            continue
    else:
        d_new['runtime_error']=d['result']['full_runtime_error']
    ds_all_new.append(d_new)
print(len(ds_all_new))

- 第二步，添加原有数据集的信息

加载原有数据集

In [ ]:
dsds=read_jsonl_gz('./leetcodedataset/LeetCodeDataset-all.jsonl.gz')
# 把原有数据集做成dict
dsds_all_dict={d['task_id']:d for d in dsds}

将原有数据集的信息添加

In [ ]:
cnt=0
ds_all_new_filter=[]
for d in ds_all_new:
    if d['task_id'] not in dsds_all_dict:
        cnt+=1
        continue
    dd=dsds_all_dict[d['task_id']]
    
    keys_to_update = ['difficulty', 'tags', 'problem_description','starter_code','completion', 'entry_point','test_cases_verify_params']
    for key in keys_to_update:
        d[key] = dd[key]
    ds_all_new_filter.append(d)
print(cnt,len(ds_all_new_filter))

- 第三步， 将数据集整合成按题目分类形式

合并lc_spec_ds

In [ ]:
ds_class={}
for d in ds_all_new_filter:
    if d['task_id'] not in ds_class:
        ds_class[d['task_id']]={}
        
        keys_to_update = ['task_id','difficulty', 'tags', 'problem_description','starter_code','completion', 'entry_point','test_cases_verify_params']
        for key in keys_to_update:
            ds_class[d['task_id']][key] = d[key]
        
        ds_class[d['task_id']]['test_cases']=[{}]
        keys_to_update = ['test_input_params', 'run_success', 'code_answer','runtime_error']
        for key in keys_to_update:
            if key in d:
                ds_class[d['task_id']]['test_cases'][0][key] = d[key]
    else:
        test_cases={}
        keys_to_update = ['test_input_params', 'run_success', 'code_answer','runtime_error']
        for key in keys_to_update:
            if key in d:
                test_cases[key] = d[key]
        ds_class[d['task_id']]['test_cases'].append(test_cases)
ds_class_list=list(ds_class.values())
print(len(ds_class_list))

将测试样例的输入输出处理成代码形式

In [ ]:
for d in ds_class_list:
    for test_case in d['test_cases']:
        test_case['test_input']=f"preconditions({test_case['test_input_params']})"
        if test_case['run_success'] is True:
            code_answer=test_case['code_answer'].replace('true','True').replace('false','False').replace('null', 'None')
            test_case['test_output']=f"postconditions({test_case['test_input_params']}, {code_answer})"
            test_case['solution_test']=f"assert {code_answer} == {d['entry_point']}({test_case['test_input_params']})"


- 保存 list of dict

In [ ]:
from utils import write_jsonl_gz
write_jsonl_gz(ds_class_list,'./lc_spec_ds.jsonl.gz')